# TrovaMUZ — Inferencia Interactiva en Tiempo Real\nCarga el adapter LoRA más reciente y genera audio sin salir de Jupyter.

In [ ]:
import sys, math
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

sys.path.insert(0, str(Path("repositories/audiocraft")))
from audiocraft.models import MusicGen

# ── Blackwell / H100 SDPA patch ──────────────────────────────────────────────
try:
    from xformers import ops as _xops
    from torch.nn import functional as _F
    def _sdpa(q, k, v, attn_bias=None, p=0.0, scale=None, **kw):
        q_t, k_t, v_t = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        is_causal, mask = False, None
        if attn_bias is not None:
            try:
                from xformers.ops.fmha.attn_bias import LowerTriangularMask
                if isinstance(attn_bias, LowerTriangularMask):
                    is_causal = True
            except Exception:
                pass
        out = _F.scaled_dot_product_attention(q_t, k_t, v_t,
            attn_mask=mask, dropout_p=p, is_causal=is_causal)
        return out.transpose(1, 2)
    _xops.memory_efficient_attention = _sdpa
    print("✓ xFormers → PyTorch SDPA patch aplicado")
except Exception as e:
    print(f"patch skipped: {e}")

# ── LoRALinear ────────────────────────────────────────────────────────────────
class LoRALinear(nn.Module):
    def __init__(self, base, r=32, alpha=32.0):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        d_in, d_out = base.in_features, base.out_features
        self.lora_A = nn.Parameter(torch.empty(r, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.lora_A.T @ self.lora_B.T)

def apply_lora(lm, r=32, alpha=32.0):
    for param in lm.parameters():
        param.requires_grad_(False)
    for name, module in list(lm.named_modules()):
        parts = name.split(".")
        if parts[-1] not in ("out_proj","linear1","linear2"):
            continue
        if not isinstance(module, nn.Linear):
            continue
        parent = lm
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], LoRALinear(module, r=r, alpha=alpha))
    return lm

print("✓ Librerías cargadas")

In [ ]:
# ── Cargar modelo (ejecuta solo una vez) ──────────────────────────────────────
ADAPTER_DIR = "models/TrovaMUZ_V1_LoRA_v3/best"   # ← cambia si usas otro

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = MusicGen.get_pretrained("melody", device=device)
apply_lora(model.lm, r=32, alpha=32.0)

def reload_adapter(path=ADAPTER_DIR):
    """Recarga el adapter sin reiniciar el modelo base — útil para probar mid-training."""
    adapter_pt = str(Path(path) / "lora_adapter.pt")
    weights = torch.load(adapter_pt, map_location=device, weights_only=False)
    model.lm.load_state_dict(weights, strict=False)
    model.lm.to(device).eval()
    print(f"✓ Adapter recargado desde {adapter_pt}")

reload_adapter()
print("✓ Modelo listo")

In [ ]:
# ── Generar y escuchar (edita el prompt y corre esta celda cuantas veces quieras) ──
PROMPT = "musicgau_adn style, TrovaMUZ_V1 Dominican bachata romantica, syncopated bicheo rhythm, requinto guitar with expressive bends, 116 BPM, A minor, romantic melancholic mood"
DURATION = 20       # segundos
TOP_K    = 250
CFG_COEF = 3.0

# Para recargar el adapter más reciente sin reiniciar todo:
# reload_adapter()

model.set_generation_params(duration=DURATION, use_sampling=True, top_k=TOP_K, cfg_coef=CFG_COEF)

with torch.no_grad():
    wav = model.generate([PROMPT], progress=True)

wav_np = wav[0, 0].cpu().numpy()
wav_np = np.clip(wav_np, -1.0, 1.0).astype(np.float32)

sf.write("output.wav", wav_np, model.sample_rate)
print(f"✓ {len(wav_np)/model.sample_rate:.1f}s generados")
display(Audio(wav_np, rate=model.sample_rate))

In [ ]:
# ── Prompts por género — copia el que quieras arriba ─────────────────────────
GENEROS = {
    "bachata":
        "musicgau_adn style, TrovaMUZ_V1 Dominican bachata romantica, syncopated bicheo rhythm, requinto guitar with expressive bends, 116 BPM, A minor, romantic melancholic mood",

    "bachata_amargue":
        "musicgau_adn style, TrovaMUZ_V1 bachata de amargue, bitter heartbreak, sharp requinto bends, punching bongo martillo, bright guira sixteenth notes, 124 BPM, D minor, raw emotional",

    "merengue":
        "musicgau_adn style, TrovaMUZ_V1 merengue tipico dominicano, accordion lead melody, tambora drum, guira scraper, fast pambiche rhythm, 160 BPM, G major, festive danceable tropical",

    "merengue_urbano":
        "musicgau_adn style, TrovaMUZ_V1 merengue urbano moderno, brass section hits, electric bass tumbao, synth pads, energetic dance floor, 170 BPM, F major, high energy urban",

    "salsa":
        "musicgau_adn style, TrovaMUZ_V1 salsa tropical, clave 2-3 rhythm, piano guajeo montuno, brass section, tumbao bass, congas and timbales, 185 BPM, F major, danceable",

    "cumbia":
        "musicgau_adn style, TrovaMUZ_V1 cumbia colombiana, gaita flute melody, caja drum, maracas, accordion, tropical groove, 120 BPM, C major, festive folk",

    "bolero":
        "musicgau_adn style, TrovaMUZ_V1 bolero romantico, warm nylon-string requinto, lush strings background, gentle bongo, intimate male vocals, 90 BPM, C major, heartfelt emotional",

    "vallenato":
        "musicgau_adn style, TrovaMUZ_V1 vallenato paseo, accordion lead, caja drum, guacharaca, storytelling vocals, 100 BPM, G major, Colombian folk nostalgic",
}

for g, p in GENEROS.items():
    print(f"[{g}]\\n  {p[:90]}...\\n")